# M3 — Priority (direct)

**Approach:** direct priority predictor (contributor C). Summary (stop-words removed, 5000 features) plus `plan_tier`, `tenure`, `channel`; class-balanced logistic regression. Evaluated on a **chronological split** (and, for reference, on a random stratified split). Ordinal extras: within-1-level and quadratic-weighted kappa. Saves `models/m3_priority_direct.joblib`.

In [ ]:
import sys, os, warnings; warnings.filterwarnings("ignore")
_p = os.getcwd()
while _p != os.path.dirname(_p) and not os.path.exists(os.path.join(_p, "northstar_prep.py")):
    _p = os.path.dirname(_p)
sys.path.insert(0, _p)
import numpy as np, pandas as pd
import northstar_prep as P

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split

df = P.load_clean()
FEATS, TARGET = ["case_summary","plan_tier","channel","customer_tenure_months"], "priority"
labels = P.PRIORITIES

def direct_pipe():
    trans = [("sum", TfidfVectorizer(max_features=5000, stop_words="english",
                                     ngram_range=(1,2), min_df=2, max_df=0.95), "case_summary"),
             P.t_cat(["plan_tier","channel"]), P.t_num(["customer_tenure_months"])]
    return P.make_clf(trans)

tr, va, te = P.time_split(df)
pipe = direct_pipe().fit(tr[FEATS], tr[TARGET])
pred = pipe.predict(te[FEATS])
base = df[TARGET].value_counts(normalize=True).iloc[0]
metrics = P.evaluate(te[TARGET], pred, labels, baseline=base)
metrics.update(P.ordinal_extras(te[TARGET], pred, labels))

rtr, rte = train_test_split(df, test_size=0.2, random_state=42, stratify=df[TARGET])
rpred = direct_pipe().fit(rtr[FEATS], rtr[TARGET]).predict(rte[FEATS])
metrics["macro_F1_random_split"] = round(float(P.f1_score(rte[TARGET], rpred, average="macro")), 3)
metrics

In [ ]:
proba = pipe.predict_proba(te[FEATS])
test = te[["case_id","case_summary","plan_tier","channel","customer_tenure_months",TARGET]].copy().reset_index(drop=True)
test["predicted"], test["confidence"] = pred, proba.max(axis=1).round(3)
full = direct_pipe().fit(df[FEATS], df[TARGET])
P.save_bundle("m3_priority_direct", {
    "model": full, "target": TARGET, "features": FEATS, "classes": labels, "kind": "multiclass",
    "label": "Priority — direct", "metrics": metrics, "test": test,
    "report": P.class_report(te[TARGET], pred, labels), "confusion": P.conf_matrix(te[TARGET], pred, labels),
    "meta": {"contributor":"C", "approach":"direct priority predictor",
             "split":"time (+random ref)", "priorities":labels,
             "inputs":"summary + plan_tier + tenure + channel"}})